# 02 - Output Schema and JSON Contract

This notebook explains the JSON contract between Member 2's Agent and Member 3's Streamlit frontend. Stable JSON matters because the UI and evaluation code should not parse free-form LLM text.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.agent.schemas.output_schema import (
    fallback_response,
    is_valid_agent_response,
    normalize_agent_response,
)

## Required Top-Level Fields

- `case_summary`
- `legal_issues`
- `legal_basis`
- `practical_references`
- `recommendations`
- `limitations`

In [ ]:
good_response = {
    "case_summary": "The user reports unpaid salary for two months.",
    "legal_issues": ["unpaid salary"],
    "legal_basis": [
        {
            "document_name": "Labor Code",
            "article": "Article 94",
            "effective_status": "Effective",
            "excerpt": "Employers must pay wages fully and on time.",
            "source": "mock://labor-code",
        }
    ],
    "practical_references": [],
    "recommendations": [
        {
            "option": "Send a written salary request",
            "when_to_use": "When salary is overdue and evidence exists.",
            "suggested_steps": ["Collect payslips", "Send written request"],
        }
    ],
    "limitations": "Decision support only.",
}

print(is_valid_agent_response(good_response))
print(json.dumps(normalize_agent_response(good_response), indent=2, ensure_ascii=False))

## Broken JSON-Like Data

LLMs sometimes omit fields or return the wrong type. The schema layer repairs basic shape problems so the frontend is protected.

In [ ]:
broken_response = {
    "case_summary": "Only one field is present.",
    "legal_issues": "this should be a list",
    "recommendations": "this should also be a list",
}

repaired = normalize_agent_response(broken_response)
print(json.dumps(repaired, indent=2, ensure_ascii=False))

## Fallback Response

When the Agent cannot safely produce a normal answer, it should still return valid JSON with a limitation message.

In [ ]:
fallback = fallback_response("My salary is unpaid.", "The LLM returned malformed JSON.")
print(json.dumps(fallback, indent=2, ensure_ascii=False))

## What You Should Understand Now

The schema file is a safety layer. It does not decide legal truth; it makes sure the final output is shaped correctly for UI and evaluation.